# 19차시 실습 — 내 프롬프트를 '자산'으로 관리하기 (Langfuse 프롬프트 관리)

> **day1 → day2 → day3 연속 실습.** day2에서 키운 내 MVP의 프롬프트를, 코드 곳곳에 흩어진 문자열이 아니라
> **버전·라벨로 관리하는 자산(PromptOps)** 으로 바꿉니다.
> 로컬 '미니 레지스트리'로 개념을 잡고, 같은 것을 **자체 호스팅 Langfuse(Docker)** 의 프롬프트 관리로 올립니다.

## 🧪 사용법
- 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다.
- **Langfuse 키가 없어도 STEP 1~2(로컬 레지스트리)로 전부 실행됩니다.** STEP 3~4는 자체 호스팅 Langfuse가 있을 때 동작합니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다. 위에서부터 순서대로 실행하세요(`Shift+Enter`).

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

day1: 프롬프트를 **코드 안에 문자열로 직접** 넣어 MVP를 만들었습니다.
day3: 같은 프롬프트를 **이름 붙은 버전 자산**으로 꺼내 — 바꾸고, 배포하고, 문제가 생기면 되돌립니다.

> 공통 예시 = **맛집 추천 도우미의 시스템 프롬프트**. 구조는 동일하니 **당신 팀 MVP의 핵심 프롬프트**로 바꾸면 그대로 적용됩니다(STEP 5).
> 핵심 질문: "프롬프트 한 줄 고쳤다가 품질이 무너졌다 — 어떻게 **안전하게** 바꾸고 **즉시 되돌릴까?**"

## STEP 1 — 로컬 미니 프롬프트 레지스트리 (`create_prompt` / `get_prompt` / `compile`)

프롬프트 관리의 핵심은 **이름 + 버전 + 라벨 + 변수**입니다. 거창한 도구 없이도
- 이름으로 등록/조회하고(`create_prompt`/`get_prompt`),
- `{{변수}}`를 채우고(`compile`),
- `production` 라벨로 "지금 쓰는 버전"을 가리키게

만들면 개념이 잡힙니다. 아래 레지스트리는 **Langfuse 프롬프트 관리의 축소판**입니다(STEP 3에서 진짜로 교체).

⌨️ **터미널 Codex:** `codex "이름·버전·production 라벨·{{변수}} compile을 지원하는 미니 프롬프트 레지스트리를 만들어줘"`

In [2]:
# 로컬 미니 프롬프트 레지스트리 — Langfuse 프롬프트 관리의 '축소판'(키 없이 개념 잡기)
PROMPTS = {}   # name -> [버전들]  (각 버전: version / prompt / config / labels)

def create_prompt(name, prompt, config=None, labels=("production",)):
    """새 '버전'을 추가한다. 같은 라벨은 최신 버전으로 이동(이전 버전에서 회수)."""
    versions = PROMPTS.setdefault(name, [])
    for v in versions:
        v["labels"] -= set(labels)
    versions.append({"version": len(versions) + 1, "prompt": prompt,
                     "config": dict(config or {}), "labels": set(labels)})
    print(f"📌 '{name}' v{versions[-1]['version']} 생성 · labels={sorted(versions[-1]['labels'])}")
    return versions[-1]

def get_prompt(name, label="production", version=None):
    """이름으로 조회 — 기본은 'production' 라벨, 또는 version 지정."""
    versions = PROMPTS[name]
    if version is not None:
        return next(v for v in versions if v["version"] == version)
    return next(v for v in reversed(versions) if label in v["labels"])

def compile_prompt(text, **kw):
    """{{변수}}를 값으로 치환 — Langfuse prompt.compile()의 축소판."""
    for k, val in kw.items():
        text = text.replace("{{" + k + "}}", str(val))
    return text

def promote(name, version, label="production"):
    """지정 버전에 라벨 부여(다른 버전에서 회수) = 배포/롤백."""
    for v in PROMPTS[name]:
        v["labels"].add(label) if v["version"] == version else v["labels"].discard(label)
    print(f"🔁 '{name}' {label} → v{version}")

print("레지스트리 준비 완료 · create_prompt / get_prompt / compile_prompt / promote")

레지스트리 준비 완료 · create_prompt / get_prompt / compile_prompt / promote


### STEP 1+ — text vs chat 프롬프트 (같은 원리, 형태만 다르다)

실제 앱은 system 하나가 아니라 **여러 메시지(system·user·assistant)** 를 씁니다.
Langfuse는 `type="chat"`으로 **메시지 리스트**를 저장하고, `compile()`이 각 메시지의 `{{변수}}`를 채웁니다 — text와 원리는 같고 **형태만** 다릅니다.

⌨️ **터미널 Codex:** `codex "system·user 두 메시지로 된 chat 프롬프트를 만들고 각 content의 변수를 compile로 채워줘"`

In [3]:
# chat 프롬프트 — 실제 앱은 system/user 여러 메시지를 쓴다 (text와 '같은 원리', 형태만 다름)
chat_prompt = [
    {"role": "system", "content": "너는 맛집 추천 도우미다. 지역 {{area}}, 예산 {{budget}}원 이하."},
    {"role": "user",   "content": "{{query}} 한 곳만 이유와 함께 추천해줘."},
]
def compile_chat(messages, **kw):
    """chat 프롬프트의 각 메시지 content에 {{변수}} 치환 — Langfuse chat prompt.compile()의 축소판."""
    return [{"role": m["role"], "content": compile_prompt(m["content"], **kw)} for m in messages]

filled = compile_chat(chat_prompt, area="강남", budget=20000, query="매운 점심")
for m in filled:
    print(f'[{m["role"]:6}] {m["content"]}')
# → text 프롬프트는 system 문자열 하나, chat 프롬프트는 이 리스트를 그대로 messages=로 넘긴다.
# → client.chat.completions.create(model=MODEL, messages=filled) 처럼 바로 사용.

[system] 너는 맛집 추천 도우미다. 지역 강남, 예산 20000원 이하.
[user  ] 매운 점심 한 곳만 이유와 함께 추천해줘.


## STEP 2 — 프롬프트를 앱에서 쓰기 + 버전 전환·롤백

앱 코드는 프롬프트 문자열을 **직접 들고 있지 않습니다.** 이름으로 **불러와서(compile) 호출**만 합니다.
그래서 프롬프트를 고쳐도 **앱 코드는 그대로** — 새 버전을 만들고 `production` 라벨만 옮기면 배포, 되돌리면 롤백입니다.

⌨️ **터미널 Codex:** `codex "프롬프트를 이름으로 불러와 변수 치환 후 호출하고, production 라벨을 옮겨 배포/롤백하는 예제를 만들어줘"`

In [4]:
# 1) v1 등록 (production)
create_prompt("recommend-system",
    "너는 맛집 추천 도우미다. {{area}}에서 예산 {{budget}}원 이하로 한 곳만 간결히 추천하라.",
    config={"model": "gpt-4o-mini", "temperature": 0.7})

def recommend(user_msg, area, budget):
    """앱 코드: 프롬프트를 '이름으로' 불러와 compile 후 호출 (문자열을 직접 들고 있지 않음)."""
    p = get_prompt("recommend-system")                    # production 버전
    system = compile_prompt(p["prompt"], area=area, budget=budget)
    resp = client.chat.completions.create(
        model=p["config"]["model"], temperature=p["config"].get("temperature", 0.7),
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user_msg}])
    return resp.choices[0].message.content.strip()

print("[v1]", recommend("매운 점심", "강남", 20000)[:90], "\n")

# 2) v2 등록 — 말투/형식만 바꿔 production이 자동으로 v2로 이동 (앱 코드는 그대로!)
create_prompt("recommend-system",
    "너는 친근한 맛집 큐레이터다. {{area}}, 예산 {{budget}}원 이하. 이모지 1개와 함께 한 곳만, 이유를 한 줄로.",
    config={"model": "gpt-4o-mini", "temperature": 0.9})
print("[v2·production]", recommend("매운 점심", "강남", 20000)[:90], "\n")

# 3) 롤백 — production을 다시 v1으로 (배포 사고 시 즉시 복구, 앱 코드 변경 없음)
promote("recommend-system", 1)
print("[롤백 후]", recommend("매운 점심", "강남", 20000)[:90])
print("현재 production 버전:", get_prompt("recommend-system")["version"])

📌 'recommend-system' v1 생성 · labels=['production']
[v1] 강남의 "부대찌개 1946"를 추천합니다. 매운 부대찌개가 인기이며, 20,000원 이하로 맛있는 점심을 즐길 수 있습니다. 

📌 'recommend-system' v2 생성 · labels=['production']
[v2·production] 강남의 '매운족발' 추천할게요🌶️ - 매콤한 족발이 입맛을 확 살려줘서 스트레스 해소에 딱이에요! 

🔁 'recommend-system' production → v1
[롤백 후] 강남의 "부대찌개 1번가"를 추천합니다. 매콤한 부대찌개를 저렴하게 즐길 수 있으며, 1인분이 8,000원 정도로 예산 내에서 충분히 맛있게 점심을 해결할 수 있
현재 production 버전: 1


## STEP 3 — Langfuse 프롬프트 관리 (자체 호스팅) — 실제 플랫폼으로

방금 만든 로컬 레지스트리는 **Langfuse 프롬프트 관리**의 축소판입니다. 실제 Langfuse는
**UI에서 프롬프트를 편집·버전 비교·라벨(production) 배포·롤백**하고, SDK로 `get_prompt`→`compile`→호출까지 이어집니다.
게다가 `langfuse_prompt=`로 링크하면 **트레이스에 '어떤 프롬프트 버전이 이 답을 만들었는지'** 가 남습니다(17·18차시 관찰성과 연결).

> **이 캠프는 Langfuse를 자체 호스팅(Docker, `http://localhost:3000`)으로 씁니다** — 0차시에서 Docker 설치.
> 띄우기: `git clone https://github.com/langfuse/langfuse.git && cd langfuse && docker compose up -d` → `localhost:3000` 가입 → 프로젝트 → **Settings ▸ API Keys** 복사 → 아래 3개 환경변수 설정 후 재실행.
> 키가 없으면 이 셀은 건너뜁니다(STEP 1~2 로컬 레지스트리로 개념은 이미 완성).

⌨️ **터미널 Codex:** `codex "자체 호스팅 Langfuse(localhost:3000)에 create_prompt로 프롬프트를 등록하고 get_prompt·compile·langfuse_prompt 링크로 호출해줘"`

In [5]:

print("Langfuse를 '자체 호스팅(Self-host)'으로 띄웁니다 (0차시에서 Docker 설치):")
print("     1) git clone https://github.com/langfuse/langfuse.git && cd langfuse")
print("     2) docker compose up -d            # http://localhost:3000 에서 열림")
print("     3) localhost:3000 가입(로컬 계정) → 프로젝트 생성 → Settings ▸ API Keys에서 키 2개 복사")
print("     4) 환경변수 설정 후 이 셀 재실행:")
print("        LANGFUSE_PUBLIC_KEY · LANGFUSE_SECRET_KEY · LANGFUSE_HOST=http://localhost:3000")
print("   참고: from langfuse.openai import openai 로 바꾸면 토큰·비용까지 자동 계측됩니다.")

# 자체 호스팅(Self-host): Docker로 직접 띄운 주소(0차시에서 Docker 설치). 기본 http://localhost:3000
from langfuse import observe, get_client
from langfuse.openai import openai 
os.environ["LANGFUSE_HOST"] = getpass.getpass("LANGFUSE_HOST (화면에 안 보임): ")
os.environ["LANGFUSE_PUBLIC_KEY"] = getpass.getpass("LANGFUSE_PUBLIC_KEY (화면에 안 보임): ")
os.environ["LANGFUSE_SECRET_KEY"] = getpass.getpass("LANGFUSE_SECRET_KEY (화면에 안 보임): ")

os.environ["LANGFUSE_DEBUG"] = "true"
lf = get_client()

# 1) 프롬프트를 Langfuse에 등록 (production 라벨 + config)
lf.create_prompt(
    name="recommend-system", type="text",
    prompt="너는 맛집 추천 도우미다. {{area}}에서 예산 {{budget}}원 이하로 한 곳만 간결히 추천하라.",
    labels=["production"],
    config={"model": "gpt-4o-mini", "temperature": 0.7})

# 2) 불러오기 → compile → 호출 (앱은 '이름'만 알면 된다)
prompt = lf.get_prompt("recommend-system")            # 기본 label="production"
system = prompt.compile(area="강남", budget=20000)

@observe()
def recommend_lf(user_msg):
    return openai.chat.completions.create(
        model=prompt.config["model"],
        temperature=prompt.config.get("temperature", 0.7),
        messages=[{"role": "system", "content": system},
                    {"role": "user", "content": user_msg}],
        langfuse_prompt=prompt,                        # ← 트레이스에 프롬프트 버전 링크
    ).choices[0].message.content

print("답변:", recommend_lf("매운 점심 한 곳 추천"))
lf.flush()
print(f"✅ {os.environ['LANGFUSE_HOST']} → Prompts 탭에서 버전/라벨, Traces에서 '이 답을 만든 프롬프트'를 확인하세요.")



Langfuse를 '자체 호스팅(Self-host)'으로 띄웁니다 (0차시에서 Docker 설치):
     1) git clone https://github.com/langfuse/langfuse.git && cd langfuse
     2) docker compose up -d            # http://localhost:3000 에서 열림
     3) localhost:3000 가입(로컬 계정) → 프로젝트 생성 → Settings ▸ API Keys에서 키 2개 복사
     4) 환경변수 설정 후 이 셀 재실행:
        LANGFUSE_PUBLIC_KEY · LANGFUSE_SECRET_KEY · LANGFUSE_HOST=http://localhost:3000
   참고: from langfuse.openai import openai 로 바꾸면 토큰·비용까지 자동 계측됩니다.


2026-07-10 22:45:37,685 - langfuse - DEBUG - Thread: Media upload consumer thread #0 started and actively processing queue items
2026-07-10 22:45:37,685 - langfuse - DEBUG - Prompt cache initialized.
2026-07-10 22:45:37,686 - langfuse - DEBUG - Startup: Score ingestion consumer thread #0 started with batch size 15 and interval 1s
2026-07-10 22:45:37,686 - langfuse - INFO - Startup: Langfuse tracer successfully initialized | public_key=pk-lf-ecd84f7b-b38a-4be5-9458-a40eb8a03784 | base_url=http://localhost:3000 | environment=default | sample_rate=1.0 | media_threads=1
2026-07-10 22:45:37,687 - langfuse - DEBUG - Creating prompt name='recommend-system', labels=['production']
2026-07-10 22:45:37,975 - langfuse - DEBUG - Getting prompt 'recommend-system-label:production'
2026-07-10 22:45:37,976 - langfuse - DEBUG - Prompt 'recommend-system-label:production' not found in cache or caching disabled.
2026-07-10 22:45:37,976 - langfuse - DEBUG - Fetching prompt 'recommend-system-label:production

답변: 강남의 "마포갈매기"를 추천합니다. 매운 양념의 갈매기살과 함께 다양한 반찬이 제공되어 맛있는 점심을 즐길 수 있습니다. 예산도 20000원 이하로 충분히 가능합니다.
✅ http://localhost:3000 → Prompts 탭에서 버전/라벨, Traces에서 '이 답을 만든 프롬프트'를 확인하세요.


### STEP 3+ — chat 프롬프트를 Langfuse에 

STEP 1+에서 로컬로 본 chat 프롬프트를, 이번엔 **Langfuse에 `type="chat"`으로 등록**합니다.
`get_prompt(...).compile(...)`은 text면 문자열, **chat이면 메시지 리스트**를 돌려줘 그대로 `messages=`로 호출에 넘깁니다 — 앱은 여전히 '이름'만 압니다.

⌨️ **터미널 Codex:** `codex "system·user 두 메시지 chat 프롬프트를 Langfuse에 type=chat으로 등록하고 compile해서 openai 호출에 넘겨줘"`

In [10]:
# chat 프롬프트를 Langfuse에 — type="chat"으로 메시지 리스트를 저장·컴파일 (키 있을 때)
lf.create_prompt(
    name="recommend-chat", type="chat",
    prompt=[
        {"role": "system", "content": "너는 맛집 추천 도우미다. 지역 {{area}}, 예산 {{budget}}원 이하."},
        {"role": "user",   "content": "{{query}} 한 곳만 이유와 함께 추천해줘."},
    ],
    labels=["production"],
    config={"model": "gpt-4o-mini", "temperature": 0.7})

chat_p = lf.get_prompt("recommend-chat", label="production", cache_ttl_seconds=0)
messages = chat_p.compile(area="강남", budget=20000, query="매운 점심")   # chat → 메시지 리스트 반환
print("compile 결과(메시지 리스트):")
for m in messages:
    print(f'  [{m["role"]:6}] {m["content"]}')

# 컴파일된 리스트를 그대로 호출에 사용 (text는 system 문자열 하나, chat은 messages 리스트)
resp = openai.chat.completions.create(
    model=chat_p.config["model"], temperature=chat_p.config.get("temperature", 0.7),
    messages=messages, langfuse_prompt=chat_p)          # 트레이스에 chat 프롬프트 버전 링크
print("\n답변:", resp.choices[0].message.content[:100])
lf.flush()

2026-07-11 00:40:12,009 - langfuse - DEBUG - Creating prompt name='recommend-chat', labels=['production']
2026-07-11 00:40:12,691 - langfuse - DEBUG - Getting prompt 'recommend-chat-label:production'
2026-07-11 00:40:12,691 - langfuse - DEBUG - Prompt 'recommend-chat-label:production' not found in cache or caching disabled.
2026-07-11 00:40:12,692 - langfuse - DEBUG - Fetching prompt 'recommend-chat-label:production' from server...


compile 결과(메시지 리스트):
  [system] 너는 맛집 추천 도우미다. 지역 강남, 예산 20000원 이하.
  [user  ] 매운 점심 한 곳만 이유와 함께 추천해줘.


2026-07-11 00:40:16,882 - langfuse - DEBUG - Trace: Processing span name='OpenAI-generation' | Full details:
{
  "name": "OpenAI-generation",
  "context": {
    "trace_id": "30b0c536097390e9f2120a57c364d5e8",
    "span_id": "959c17572f417133",
    "trace_state": "[]"
  },
  "kind": "SpanKind.INTERNAL",
  "parent_id": null,
  "start_time": "2026-07-10T15:40:12.766019Z",
  "end_time": "2026-07-10T15:40:16.881607Z",
  "status": {
    "status_code": "UNSET"
  },
  "attributes": {
    "langfuse.internal.is_app_root": true,
    "langfuse.observation.input": "[{\"role\": \"system\", \"content\": \"\\ub108\\ub294 \\ub9db\\uc9d1 \\ucd94\\ucc9c \\ub3c4\\uc6b0\\ubbf8\\ub2e4. \\uc9c0\\uc5ed \\uac15\\ub0a8, \\uc608\\uc0b0 20000\\uc6d0 \\uc774\\ud558.\"}, {\"role\": \"user\", \"content\": \"\\ub9e4\\uc6b4 \\uc810\\uc2ec \\ud55c \\uacf3\\ub9cc \\uc774\\uc720\\uc640 \\ud568\\uaed8 \\ucd94\\ucc9c\\ud574\\uc918.\"}]",
    "langfuse.observation.prompt.name": "recommend-chat",
    "langfuse.observation.pr


답변: 강남에서 매운 점심을 원하신다면 '이태원 부대찌개'를 추천합니다. 이곳은 부대찌개를 전문으로 하는 맛집으로, 매운 국물과 다양한 재료가 어우러져 깊은 맛을 자랑합니다. 

**이유


2026-07-11 00:40:17,162 - langfuse - DEBUG - Successfully flushed OTEL tracer provider
2026-07-11 00:40:17,163 - langfuse - DEBUG - Successfully flushed score ingestion queue
2026-07-11 00:40:17,163 - langfuse - DEBUG - Successfully flushed media upload queue


## STEP 4 — 여러 버전 쌓기 · 롤백 · 캐시 (운영 관점, 노트북에서 직접 확인)

프롬프트 관리의 진짜 값은 **운영 중 안전하게 바꾸기**입니다. UI 설명만 듣지 말고, 여기서 SDK로 직접 해봅니다.
- **여러 버전**: 같은 이름으로 `create_prompt`를 다시 부르면 버전이 v1·v2·v3…로 쌓이고 `production` 라벨은 최신으로 이동
- **롤백**: `update_prompt(name, version=이전번호, new_labels=["production"])` 로 **production 라벨을 이전 버전으로 이동** → 앱 재배포 없이 즉시 복구 (UI의 라벨 드래그와 같은 동작)
- **캐시**: `get_prompt(..., cache_ttl_seconds=60)` — 아래에서 **콜드 vs 웜 지연을 재서** 캐시가 실제로 네트워크 왕복을 없애는지 눈으로 확인
- **폴백**: Langfuse가 안 떠 있어도 앱이 죽지 않도록 기본 프롬프트를 넘긴다

⌨️ **터미널 Codex:** `codex "create_prompt로 버전을 여러 개 쌓고 update_prompt로 production 라벨을 이전 버전으로 롤백한 뒤, cache_ttl_seconds로 콜드/웜 지연을 비교해줘"`

In [6]:
# 여러 버전 쌓기 · 롤백 · 캐시 · 폴백 — 자체 호스팅 + 키가 있을 때만 실행
import time
from langfuse import get_client
lf = get_client()

# 1) 여러 버전 만들기 — 같은 이름으로 create_prompt를 다시 부르면 버전이 쌓이고 production 라벨은 '최신'으로 이동
lf.create_prompt(name="recommend-system", type="text",
    prompt="너는 맛집 추천 도우미다. {{area}}에서 예산 {{budget}}원 이하로 한 곳만 간결히 추천하라.",
    labels=["production"], config={"model": "gpt-4o-mini", "temperature": 0.7})
lf.create_prompt(name="recommend-system", type="text",
    prompt="너는 친근한 맛집 큐레이터다. {{area}}, 예산 {{budget}}원 이하. 이모지 1개와 이유 한 줄로 한 곳만.",
    labels=["production"], config={"model": "gpt-4o-mini", "temperature": 0.9})

lf.clear_prompt_cache()
latest = lf.get_prompt("recommend-system", label="production", cache_ttl_seconds=0)
print(f"현재 production 버전: v{latest.version}  ·  프롬프트: {latest.prompt[:34]}...")

# 2) 롤백 — production 라벨을 '이전 버전'으로 이동 (앱 코드/재배포 없이 SDK 한 줄 = UI 라벨 이동과 동일)
rollback_to = latest.version - 1
lf.update_prompt(name="recommend-system", version=rollback_to, new_labels=["production"])
lf.clear_prompt_cache()  # 라벨이 바뀌었으니 캐시 무효화 후 다시 조회
after = lf.get_prompt("recommend-system", label="production", cache_ttl_seconds=0)
print(f"롤백 후  production 버전: v{after.version}  ·  프롬프트: {after.prompt[:34]}...")
assert after.version == rollback_to, "production 라벨이 이전 버전으로 이동해야 정상"

# 3) 캐시가 '실제로' 되는지 — 콜드(서버 왕복) vs 웜(캐시 히트) 지연을 재서 노트북에서 눈으로 확인
lf.clear_prompt_cache()
t0 = time.perf_counter(); lf.get_prompt("recommend-system", cache_ttl_seconds=60); cold = (time.perf_counter() - t0) * 1000
t1 = time.perf_counter(); lf.get_prompt("recommend-system", cache_ttl_seconds=60); warm = (time.perf_counter() - t1) * 1000
print(f"\n캐시 콜드(서버 왕복): {cold:8.2f} ms")
print(f"캐시 웜 (캐시 히트) : {warm:8.2f} ms   → 약 {cold / max(warm, 1e-6):,.0f}배 빠름 = 네트워크 왕복이 사라짐")

# 4) 폴백 — Langfuse가 없거나 이름이 없어도 앱이 죽지 않게 기본 프롬프트 사용
safe = lf.get_prompt("recommend-system",
    fallback="너는 맛집 추천 도우미다. {{area}} 예산 {{budget}}원 이하로 추천하라.")
print("\n폴백 포함 compile:", safe.compile(area="홍대", budget=15000)[:50], "...")

2026-07-10 22:45:40,263 - langfuse - DEBUG - Creating prompt name='recommend-system', labels=['production']
2026-07-10 22:45:40,299 - langfuse - DEBUG - Creating prompt name='recommend-system', labels=['production']
2026-07-10 22:45:40,326 - langfuse - DEBUG - Getting prompt 'recommend-system-label:production'
2026-07-10 22:45:40,326 - langfuse - DEBUG - Prompt 'recommend-system-label:production' not found in cache or caching disabled.
2026-07-10 22:45:40,327 - langfuse - DEBUG - Fetching prompt 'recommend-system-label:production' from server...
2026-07-10 22:45:40,390 - langfuse - DEBUG - Getting prompt 'recommend-system-label:production'
2026-07-10 22:45:40,390 - langfuse - DEBUG - Prompt 'recommend-system-label:production' not found in cache or caching disabled.
2026-07-10 22:45:40,391 - langfuse - DEBUG - Fetching prompt 'recommend-system-label:production' from server...
2026-07-10 22:45:40,409 - langfuse - DEBUG - Getting prompt 'recommend-system-label:production'
2026-07-10 22:45

현재 production 버전: v4  ·  프롬프트: 너는 친근한 맛집 큐레이터다. {{area}}, 예산 {{bu...
롤백 후  production 버전: v3  ·  프롬프트: 너는 맛집 추천 도우미다. {{area}}에서 예산 {{bud...

캐시 콜드(서버 왕복):     4.27 ms
캐시 웜 (캐시 히트) :     0.21 ms   → 약 20배 빠름 = 네트워크 왕복이 사라짐

폴백 포함 compile: 너는 맛집 추천 도우미다. 홍대에서 예산 15000원 이하로 한 곳만 간결히 추천하라. ...


## STEP 5 — Champion vs Challenger: 느낌이 아니라 점수로 승격 (Langfuse)

방금 세팅한 **Langfuse의 두 버전**(production=champion, latest=challenger)을 같은 입력으로 돌립니다.
`langfuse.openai`로 **호출을 자동 트레이스**하고 `langfuse_prompt=`로 어떤 버전이 답했는지 링크한 뒤,
**점수가 높은 쪽만** `update_prompt`로 `production` 라벨을 옮겨 승격합니다 — 진 버전은 배포 금지.

> STEP 3·4(Langfuse 키)가 실행돼 있어야 합니다. 버전이 하나뿐이면 champion=challenger가 됩니다.

⌨️ **터미널 Codex:** `codex "Langfuse의 production과 latest 프롬프트를 같은 입력으로 돌려 점수를 비교하고, 이긴 버전으로 production 라벨을 옮겨줘"`

In [7]:
# Champion vs Challenger — Langfuse 두 버전을 점수로 비교 후 라벨 승격 (키 있을 때)
from langfuse import get_client
from langfuse.openai import openai          # 호출이 자동으로 Langfuse 트레이스로 남는다
lf = get_client()

TEST_INPUTS = ["강남 2만원 매운 점심", "홍대 혼밥 저녁", "역삼 회식 장소"]
def quick_score(text):                       # 간단 규칙 채점: 가격·지역 언급 여부
    return ("원" in text) + any(a in text for a in ["강남", "홍대", "역삼"])

def score_prompt(p):
    total = 0
    for q in TEST_INPUTS:
        system = p.compile(area=q.split()[0], budget=20000)
        r = openai.chat.completions.create(
            model=p.config.get("model", "gpt-4o-mini"),
            temperature=p.config.get("temperature", 0.7),
            messages=[{"role": "system", "content": system}, {"role": "user", "content": q}],
            langfuse_prompt=p)               # ← 트레이스에 '이 답을 만든 프롬프트 버전' 링크
        total += quick_score(r.choices[0].message.content)
    return total / len(TEST_INPUTS)

champ = lf.get_prompt("recommend-system", label="production", cache_ttl_seconds=0)
chall = lf.get_prompt("recommend-system", label="latest",     cache_ttl_seconds=0)   # 최신 버전
sc, sh = score_prompt(champ), score_prompt(chall)
print(f"champion v{champ.version}={sc:.2f}  vs  challenger v{chall.version}={sh:.2f}")

winner = chall if sh > sc else champ
lf.update_prompt(name="recommend-system", version=winner.version, new_labels=["production"])  # 이긴 버전만 승격
lf.clear_prompt_cache()
lf.flush()
print(f"→ v{winner.version}를 production으로 승격 (Langfuse 라벨 이동, 진 버전은 배포 금지)")
print("   Traces 탭에서 각 호출이 어떤 프롬프트 버전으로 실행됐는지 확인하세요.")

2026-07-10 22:45:40,420 - langfuse - DEBUG - Getting prompt 'recommend-system-label:production'
2026-07-10 22:45:40,420 - langfuse - DEBUG - Prompt 'recommend-system-label:production' not found in cache or caching disabled.
2026-07-10 22:45:40,421 - langfuse - DEBUG - Fetching prompt 'recommend-system-label:production' from server...
2026-07-10 22:45:40,448 - langfuse - DEBUG - Getting prompt 'recommend-system-label:latest'
2026-07-10 22:45:40,450 - langfuse - DEBUG - Prompt 'recommend-system-label:latest' not found in cache or caching disabled.
2026-07-10 22:45:40,451 - langfuse - DEBUG - Fetching prompt 'recommend-system-label:latest' from server...
2026-07-10 22:45:42,060 - langfuse - DEBUG - Trace: Processing span name='OpenAI-generation' | Full details:
{
  "name": "OpenAI-generation",
  "context": {
    "trace_id": "09576a2560b94988c44c9cba5ece0200",
    "span_id": "9534eae439277751",
    "trace_state": "[]"
  },
  "kind": "SpanKind.INTERNAL",
  "parent_id": null,
  "start_time":

champion v3=2.00  vs  challenger v4=0.67
→ v3를 production으로 승격 (Langfuse 라벨 이동, 진 버전은 배포 금지)
   Traces 탭에서 각 호출이 어떤 프롬프트 버전으로 실행됐는지 확인하세요.


## STEP 6 — 프롬프트 드리프트: 내가 안 바꿔도 무너진다 (Langfuse)

모델·설정이 바뀌면 같은 프롬프트도 성능이 떨어질 수 있습니다. Langfuse의 **production 프롬프트**를 그대로 쓰되
'모델이 흔들리는' 상황(temperature 급변)을 시뮬레이션해 **STEP 5의 기준선과 비교**합니다. 호출은 모두 Langfuse 트레이스로 남습니다.

In [8]:
# 프롬프트 드리프트 — 같은 production 프롬프트, '흔들리는 모델'이면 점수가 떨어진다 (기준선 대비)
BASELINE_SCORE = max(sc, sh)                 # STEP 5의 승자 점수 = 성능 기준선
prod = lf.get_prompt("recommend-system", label="production", cache_ttl_seconds=0)

def run_drifted():
    total = 0
    for q in TEST_INPUTS:
        system = prod.compile(area=q.split()[0], budget=20000)
        r = openai.chat.completions.create(
            model=prod.config.get("model", "gpt-4o-mini"),
            temperature=1.9,                 # '모델 업데이트'로 흔들리는 상황 시뮬레이션
            messages=[{"role": "system", "content": system}, {"role": "user", "content": q}],
            langfuse_prompt=prod)
        total += quick_score(r.choices[0].message.content)
    return total / len(TEST_INPUTS)

drifted = run_drifted()
lf.flush()
print(f"기준선 {BASELINE_SCORE:.2f} → 현재 {drifted:.2f}")
print("🚨 프롬프트 드리프트 감지 — 프롬프트 재작업·재평가 필요" if drifted < BASELINE_SCORE * 0.8 else "✅ 기준선 유지")
# 실전: 모델 버전이 바뀔 때마다 이 비교를 정기 실행하고, Traces/Scores로 추세를 본다

2026-07-10 22:45:48,892 - langfuse - DEBUG - Getting prompt 'recommend-system-label:production'
2026-07-10 22:45:48,892 - langfuse - DEBUG - Prompt 'recommend-system-label:production' not found in cache or caching disabled.
2026-07-10 22:45:48,893 - langfuse - DEBUG - Fetching prompt 'recommend-system-label:production' from server...
2026-07-10 22:45:58,641 - langfuse - DEBUG - Trace: Processing span name='OpenAI-generation' | Full details:
{
  "name": "OpenAI-generation",
  "context": {
    "trace_id": "d8585bb165b3ae3b4c7e7d020291fccb",
    "span_id": "2217476564e7ceda",
    "trace_state": "[]"
  },
  "kind": "SpanKind.INTERNAL",
  "parent_id": null,
  "start_time": "2026-07-10T13:45:48.903256Z",
  "end_time": "2026-07-10T13:45:58.570507Z",
  "status": {
    "status_code": "UNSET"
  },
  "attributes": {
    "langfuse.internal.is_app_root": true,
    "langfuse.observation.input": "[{\"role\": \"system\", \"content\": \"\\ub108\\ub294 \\ub9db\\uc9d1 \\ucd94\\ucc9c \\ub3c4\\uc6b0\\ubbf8

기준선 2.00 → 현재 1.33
🚨 프롬프트 드리프트 감지 — 프롬프트 재작업·재평가 필요


## STEP 7 — ⭐ 내 MVP에 적용

당신 팀 MVP의 **핵심 프롬프트 한 개**를 골라 자산으로 관리하세요.
1. 프롬프트에 **변수(`{{...}}`)** 를 넣어 이름을 붙인다 (예: `recommend-system`, `summarize-review`)
2. 로컬 레지스트리(STEP1) 또는 Langfuse(STEP3)에 등록하고, 앱은 **이름으로만** 불러 쓴다
3. 개선안을 새 버전으로 올리고 `production` 라벨로 배포 — 문제가 생기면 롤백

⌨️ **터미널 Codex:** `codex "우리 MVP의 핵심 프롬프트를 변수화해 Langfuse에 등록하고 앱이 이름으로 불러 쓰도록 바꿔줘"`

In [9]:
# 팀별 작성: 우리 MVP에서 '자산으로 관리할' 프롬프트 1개
MY_PROMPT = {
    # "name": "summarize-review",
    # "prompt": "다음 리뷰를 {{tone}} 말투로 3문장 요약: {{review}}",
    # "config": {"model": "gpt-4o-mini", "temperature": 0.5},
    # "vars": {"tone": "친근한", "review": "여기에 실제 리뷰"},
}
if MY_PROMPT:
    create_prompt(MY_PROMPT["name"], MY_PROMPT["prompt"], config=MY_PROMPT.get("config"))
    p = get_prompt(MY_PROMPT["name"])
    print("compile 결과:\n", compile_prompt(p["prompt"], **MY_PROMPT.get("vars", {})))
    print("\n→ 저녁 팀 프로젝트: 이 프롬프트를 Langfuse(자체 호스팅)에 올리고 버전·라벨로 운영하세요.")
else:
    print("⬜ MY_PROMPT를 채운 뒤 다시 실행하세요 — 우리 MVP의 첫 '관리되는 프롬프트'")

⬜ MY_PROMPT를 채운 뒤 다시 실행하세요 — 우리 MVP의 첫 '관리되는 프롬프트'


## 정리

- **프롬프트는 코드에 박는 문자열이 아니라 버전·라벨로 관리하는 자산(PromptOps)** — 앱은 이름으로만 불러 쓴다.
- Langfuse 프롬프트 관리: `create_prompt`(text/chat·config·labels) → `get_prompt`(label/version/cache/fallback) → `compile({{변수}})`.
- `langfuse_prompt=`로 링크하면 **트레이스에 '어떤 프롬프트 버전이 답했는지'** 가 남아 17·18차시 관찰성과 이어진다.
- **배포 = production 라벨 이동, 롤백 = 라벨 되돌리기** — 앱 재배포 없이 즉시. 폴백으로 장애에도 앱이 죽지 않게.
- 오늘 만든 프롬프트 자산은 **저녁 팀 프로젝트**에 그대로 적용하세요.
- 다음 차시(20): 바꾼 프롬프트가 **정말 더 나은지 숫자로 평가**한다 — Langfuse 평가(Datasets·Scores·LLM-as-judge).